# Лабораторная работа #11. Пространственная фильтрация, выделение простейших признаков. Бинаризация изображений.

В прошлой лабораторной мы поработали с простейшими методами обработки изображений. Теперь пришло время использовать изображения для каких-то задач.

Конечно, можно просто загонять все изображения всегда целиком в нейросетку. Но это связано с большими проблемами (ведь изображения разные). Мы можем помочь методам обучения, обработав изображение особым универсальным образом. Например, удобно убрать шумы, выделить детали, контуры. Этим и займёмся сегодня.

## Памятка по opencv

```python
# Лапласиан
laplacian = cv2.Laplacian(img, cv2.CV_64F)

# Собель X и Y
sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)  # горизонтальный
sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)  # вертикальный

# Гауссово размытие
blurred = cv2.GaussianBlur(img, (5,5), 0)  # (ksize, sigma)

# Ручной порог
_, binary_manual = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

# Метод Оцу
_, binary_otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Поиск количества связных областей
count, _ = cv2.connectedComponents(img)
```

## Загрузка изображения

In [2]:
# Прочитаем картинку сразу в оттенках серого.
# Работа с цветным изображением такая же, как и в сером, просто кода в 3 раза больше (для каждого канала).
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

img = cv2.imread('Alonso_better_than_Kimi.jpg')
rgbimg = cv2.imread('Alonso_better_than_Kimi.jpg', cv2.IMREAD_COLOR_RGB)
grimg = cv2.imread('Alonso_better_than_Kimi.jpg', cv2.IMREAD_GRAYSCALE)
cv2_imshow(grimg)
grimg

ModuleNotFoundError: No module named 'cv2'

## Шумоподавление

In [ ]:
# Внесём шума нормально распределенного случайного шума в картинку. Неужели вы думали, что всё будет так просто?
noise = np.random.normal(12, 12, grimg.shape).astype(np.uint16) # создает массив случайных чисел где 0 среднее значение а 5 стандартное отклонение

# Преобразуем в int16 (-32768 до 32767) тк unit8 (который стоит по дефолту) имеет диапозон 0-255 (т.е. 256 чисел) а дальше оно залупливается (100 + 200 = 44)
noisy = np.clip(grimg.astype(np.int16) + noise, 0, 255).astype(np.uint8)
# тут мы приабвляем 16битные числа, клипом обрезаем их в диапозон 0-255 и возвращаем в 8битные с помощью .astype(np.uint8)
cv2_imshow(noisy)

In [ ]:
gaussian = cv2.GaussianBlur(grimg, (91, 91), 1)
cv2_imshow(gaussian)

median = cv2.medianBlur(grimg, 3)
cv2_imshow(median)

## Выделение деталей

### С помощью Гауссова фильтра

Что делает Гауссов фильтр? Он размывает картинку. Что такое размытие? Убирание деталей.

Тогда как найти детали? Вычесть из исходного размытое!

In [ ]:
gauss_difference = cv2.subtract(grimg, cv2.GaussianBlur(grimg, (9, 9), 1.5))
cv2_imshow(gauss_difference)
print(gauss_difference)

In [ ]:
cv2_imshow(img + gauss_difference)
cv2_imshow(img)

### С помощью Лапласиана



Есть и другие способы выделить детали. Например, использовать особого вида свёртку - Лапласиан. Определяется Лапласиан производными второго порядка и способен выделять мелкие детали, изгибы и впадины функции изменения яркости.

In [ ]:
# TODO: применить Лапласиан к исходному изображению.
# ALERT: После применения Лапласиана следует нормализовать картинку!

# laplacian = cv2.Laplacian(img, cv2.CV_64F) не работает с ИСХОДНЫМ изображением, ни с BGR которое по умолчанию в cv2.imshow, ни с RGB которое по умолчанию в cv2_imshow
b, g, r = cv2.split(img)

laplacian_b = cv2.Laplacian(b, cv2.CV_64F)
laplacian_g = cv2.Laplacian(g, cv2.CV_64F)
laplacian_r = cv2.Laplacian(r, cv2.CV_64F)

laplacian_b_norm = cv2.normalize(laplacian_b, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
laplacian_g_norm = cv2.normalize(laplacian_g, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
laplacian_r_norm = cv2.normalize(laplacian_r, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

laplacian_rgb = cv2.merge([laplacian_b_norm, laplacian_g_norm, laplacian_r_norm])
cv2_imshow(laplacian_rgb)


laplacian_gray = cv2.Laplacian(grimg, cv2.CV_64F)
laplacian_norm = cv2.normalize(laplacian_gray, None, 0, 1, cv2.NORM_MINMAX)
laplacian_norm

In [ ]:
# TODO: прибавить Лапласиан к исходной картинке. Стало ли изображение чётче?
cv2_imshow(grimg)
cv2_imshow(grimg + laplacian_norm)

## Выделение контуров с помощью фильтров Собеля

Контуры изображения фактически являются просто местами большого перепада яркостей пикселей. Похоже на первую производную. На ней основаны фильтры Собеля. Их два: горизонтальный и вертикальный (соответственно для перепадов по вертикали и по горизонтали). Часто их объединяют или корнем из суммы квадратов, или просто модулем суммы. Мы используем второй вариант.

In [ ]:
# TODO: посчитать фильтр Собеля по вертикали, по горизонтали. Получить единый результат модулем суммы
# ALERT: после модуля суммы не забудьте нормализовать!


# TODO: посчитать фильтр Собеля по вертикали, по горизонтали. Получить единый результат модулем суммы
# ALERT: после модуля суммы не забудьте нормализовать!

sobel_x = cv2.Sobel(grimg, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(grimg, cv2.CV_64F, 0, 1, ksize=3)


# Абсолютное значение (так как производная может быть отрицательной (Фильтр Собеля вычисляет приближенную производную изображения, показывая, где яркость меняется быстрее всего - это и есть границы.))
sobel_x_abs = cv2.convertScaleAbs(sobel_x)
sobel_y_abs = cv2.convertScaleAbs(sobel_y)

sobel_sum_abs = np.abs(sobel_x_abs + sobel_y_abs)
sobel_result = cv2.addWeighted(sobel_x_abs, 0.5, sobel_y_abs, 0.5, 0)

cv2_imshow(sobel_x_abs)
cv2_imshow(sobel_y_abs)
cv2_imshow(sobel_result)

## Работа с бинарными изображениями

Иногда нам вовсе не нужны детали. Нам достаточно знать контуры, очертания объектов. Например, для того, чтобы посчитать их количество. Попробуем бинаризовать картинку разными способами и посчитать число объектов на картинке.

### Ручная бинаризация

Самый тривиальный, но иногда очень точный метод.

In [ ]:
# TODO: постройте гистограмму изображения.
# TIP: картинку можно развернуть в одномерный массив методом ravel().
# Как построить гистограмму по одномерному массиву мы смотрели на предыдущих лабораторных

rav = grimg.ravel()
# .ravel() преобразует многомерный массив в 1D массив, например (3, 3) в (9,). Не копирует данные (изменения в ravel-массиве влияют на исходный)

# Строим гистограммы для каждого канала
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.imshow(grimg, cmap='gray')
plt.title('Серое изображение')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.hist(rav, bins=256, range=(0, 255), alpha=0.5, color='gray', label='gray')
plt.title('Гистограммы серого каналов')
plt.xlabel('Значение пикселя')
plt.ylabel('Количество пикселей')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# TODO: по гистограмме выберите примерно пороговое значение (минимум между двумя пиками)
# Бинаризуйте картинку по пороговому значению с помощью cv2.threshold


plt.subplot(1, 2, 1)
plt.imshow(grimg, cmap='gray')
plt.axis('off')
_, binary_manual = cv2.threshold(grimg, 47, 255, cv2.THRESH_BINARY)
plt.subplot(1, 2, 2)
plt.imshow(binary_manual, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
count, _ = cv2.connectedComponents(binary_manual)

print(count)

### Автоматическая бинаризация методом Оцу.

Метод Оцу – популярный математический метод бинаризации. Основан на минимизации и максимизации дисперсии (отклонения от среднего) внутри областей гистограммы и между ними. Встроен в opencv, так что реализовывать его не придётся :)

In [3]:
# TODO: # бинаризуйте картинку методом Оцу с помощью cv2.threshold

_, binary_otsu = cv2.threshold(grimg, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
cv2_imshow(binary_otsu)
print(_)

NameError: name 'cv2' is not defined

In [ ]:
count, _ = cv2.connectedComponents(binary_otsu)
print(count)

### Предобработка картинки

In [ ]:
# TODO: попробуйте бинаризовать зашумленное изображение из раздела "Шумоподавление"
# Рядом бинаризуйте изображение, в котором шумы подавлены фильтром по Гауссу
# Сравните, сколько объектов находит cv2.connectedComponents в обоих случаях

noisy2 = cv2.GaussianBlur(noisy, (91, 91), 1)

_, otsu_task = cv2.threshold(noisy, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
_, otsu_n = cv2.threshold(noisy2, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

count2, _ = cv2.connectedComponents(otsu_task)
count3, _ = cv2.connectedComponents(otsu_n)

print(count2)
print(count3)


plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(otsu_task, cmap='gray')
plt.title('Серое изображение')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.hist(otsu_task.ravel(), bins=256, range=(0, 255), alpha=0.5, color='gray', label='gray')
plt.title('Гистограммы серого каналов')
plt.xlabel('Значение пикселя')
plt.ylabel('Количество пикселей')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.hist(noisy.ravel(), bins=256, range=(0, 255), alpha=0.5, color='gray', label='gray')
plt.title('Гистограммы серого каналов')
plt.xlabel('Значение пикселя')
plt.ylabel('Количество пикселей')
plt.grid(True, alpha=0.3)


plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(otsu_n, cmap='gray')
plt.title('Серое изображение')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.hist(otsu_n.ravel(), bins=256, range=(0, 255), alpha=0.5, color='gray', label='gray')
plt.title('Гистограммы серого каналов')
plt.xlabel('Значение пикселя')
plt.ylabel('Количество пикселей')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.hist(noisy2.ravel(), bins=256, range=(0, 255), alpha=0.5, color='gray', label='gray')
plt.title('Гистограммы серого каналов')
plt.xlabel('Значение пикселя')
plt.ylabel('Количество пикселей')
plt.grid(True, alpha=0.3)


#